In [1]:
import random
import numpy as np
import torch

from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES
import gradio as gr

import openai
import whisper

import soundfile as sf
import requests

import os
import glob
import re

c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chatterbox TTS (WIP)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Running on device: {DEVICE}")

# --- Global Model Initialization ---
MODEL = None

LANGUAGE_CONFIG = {
    "en": {
        "audio": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/en_f1.flac",
        "text": "Last month, we reached a new milestone with two billion views on our YouTube channel."
    },
    "ru": {
        "audio": "https://storage.googleapis.com/chatterbox-demo-samples/mtl_prompts/ru_m.flac",
        "text": "В прошлом месяце мы достигли нового рубежа: два миллиарда просмотров на нашем YouTube-канале."
    },
    
}

# --- UI Helpers ---
def default_audio_for_ui(lang: str) -> str | None:
    return LANGUAGE_CONFIG.get(lang, {}).get("audio")


def default_text_for_ui(lang: str) -> str:
    return LANGUAGE_CONFIG.get(lang, {}).get("text", "")

def on_language_change(lang, current_ref, current_text):
            return default_audio_for_ui(lang), default_text_for_ui(lang)


def get_supported_languages_display() -> str:
    """Generate a formatted display of all supported languages."""
    language_items = []
    for code, name in sorted(SUPPORTED_LANGUAGES.items()):
        language_items.append(f"**{name}** (`{code}`)")
    
    # Split into 2 lines
    mid = len(language_items) // 2
    line1 = " • ".join(language_items[:mid])
    line2 = " • ".join(language_items[mid:])
    
    return f"""
### 🌍 Supported Languages ({len(SUPPORTED_LANGUAGES)} total)
{line1}

{line2}
"""

def get_or_load_model():
    """Loads the ChatterboxMultilingualTTS model if it hasn't been loaded already,
    and ensures it's on the correct device."""
    global MODEL
    if MODEL is None:
        print("Model not loaded, initializing...")
        try:
            MODEL = ChatterboxMultilingualTTS.from_pretrained(DEVICE)
            if hasattr(MODEL, 'to') and str(MODEL.device) != DEVICE:
                MODEL.to(DEVICE)
            print(f"Model loaded successfully. Internal device: {getattr(MODEL, 'device', 'N/A')}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise
    return MODEL

def set_seed(seed: int):
    """Sets the random seed for reproducibility across torch, numpy, and random."""
    torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)

def resolve_audio_prompt(language_id: str, provided_path: str | None) -> str | None:
    """
    Decide which audio prompt to use:
    - If user provided a path (upload/mic/url), use it.
    - Else, fall back to language-specific default (if any).
    """
    if provided_path and str(provided_path).strip():
        return provided_path
    return LANGUAGE_CONFIG.get(language_id, {}).get("audio")


def generate_tts_audio(
    text_input: str,
    language_id: str,
    audio_prompt_path_input: str = None,
    exaggeration_input: float = 0.5,
    temperature_input: float = 0.8,
    seed_num_input: int = 0,
    cfgw_input: float = 0.5
) -> tuple[int, np.ndarray]:
    """
    Generate high-quality speech audio from text using Chatterbox Multilingual model with optional reference audio styling.
    Supported languages: English, French, German, Spanish, Italian, Portuguese, and Hindi.
    
    This tool synthesizes natural-sounding speech from input text. When a reference audio file 
    is provided, it captures the speaker's voice characteristics and speaking style. The generated audio 
    maintains the prosody, tone, and vocal qualities of the reference speaker, or uses default voice if no reference is provided.

    Args:
        text_input (str): The text to synthesize into speech (maximum 300 characters)
        language_id (str): The language code for synthesis (eg. en, fr, de, es, it, pt, hi)
        audio_prompt_path_input (str, optional): File path or URL to the reference audio file that defines the target voice style. Defaults to None.
        exaggeration_input (float, optional): Controls speech expressiveness (0.25-2.0, neutral=0.5, extreme values may be unstable). Defaults to 0.5.
        temperature_input (float, optional): Controls randomness in generation (0.05-5.0, higher=more varied). Defaults to 0.8.
        seed_num_input (int, optional): Random seed for reproducible results (0 for random generation). Defaults to 0.
        cfgw_input (float, optional): CFG/Pace weight controlling generation guidance (0.2-1.0). Defaults to 0.5, 0 for language transfer. 

    Returns:
        tuple[int, np.ndarray]: A tuple containing the sample rate (int) and the generated audio waveform (numpy.ndarray)
    """
    current_model = get_or_load_model()

    if current_model is None:
        raise RuntimeError("TTS model is not loaded.")

    if seed_num_input != 0:
        set_seed(int(seed_num_input))

    print(f"Generating audio for text: '{text_input[:50]}...'")
    
    # Handle optional audio prompt
    chosen_prompt = audio_prompt_path_input or default_audio_for_ui(language_id)
    
    try:
        cfgw_value = float(cfgw_input) if not isinstance(cfgw_input, (int, float)) else cfgw_input
        exag_value = float(exaggeration_input) if not isinstance(exaggeration_input, (int, float)) else exaggeration_input
    except:
        cfgw_value = 0.5
        exag_value = 0.5

    generate_kwargs = {
        "exaggeration": exag_value,
        "temperature": temperature_input,
        "cfg_weight": cfgw_value,
    }
    if chosen_prompt:
        generate_kwargs["audio_prompt_path"] = chosen_prompt
        print(f"Using audio prompt: {chosen_prompt}")
    else:
        print("No audio prompt provided; using default voice.")
        
    wav = current_model.generate(
        text_input[:300],  # Truncate text to max chars
        language_id=language_id,
        **generate_kwargs
    )
    print("Audio generation complete.")
    return (current_model.sr, wav.squeeze(0).numpy())

🚀 Running on device: cuda


loaded PerthNet (Implicit) at step 250,000


Model loaded successfully. Internal device: cuda
Generating audio for text: 'Привет! 😊 Очень рад тебя видеть! Как дела? Что бы ...'
Using audio prompt: ru_m.flac


C:\Users\salni\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\transformers\generation\configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_attentions` is. When `return_dict_in_generate` is not `True`, `output_attentions` is ignored.
  warnings.warn(
Sampling:  10%|▉         | 95/1000 [00:03<00:34, 25.92it/s]

In [3]:
#audio = generate_tts_audio("Привет", "ru", 'ru_m.flac')
#sf.write('output.wav', audio[1], samplerate=audio[0])

# Whisper AI

In [4]:
whisper.available_models()

['tiny.en',
 'tiny',
 'base.en',
 'base',
 'small.en',
 'small',
 'medium.en',
 'medium',
 'large-v1',
 'large-v2',
 'large-v3',
 'large',
 'large-v3-turbo',
 'turbo']

In [5]:
# model can be passed as dropdown in gradio UI
whisper_model = whisper.load_model("base")
whisper_model.device

device(type='cuda', index=0)

In [6]:
import tempfile


def transcribe_audio(audio_file):
    """Преобразование аудио в текст с помощью Whisper"""
    try:
        # Проверяем, что аудио было получено
        if audio_file is None:
            return "Не получено аудио для распознавания"
        
        print(f"Получен аудио вход: {type(audio_file)}")
        
        # Gradio может вернуть либо путь к файлу, либо numpy array
        if isinstance(audio_file, str):
            # Это путь к файлу
            audio_path = audio_file
            print(f"Путь к аудиофайлу: {audio_path}")
            
            # Проверяем существование файла
            if not os.path.exists(audio_path):
                return f"Файл не найден: {audio_path}"
            
            # Транскрибируем
            result = whisper_model.transcribe(audio_path, language="ru")
            return result["text"]
            
        elif isinstance(audio_file, tuple) and len(audio_file) == 2:
            # Это кортеж (sample_rate, audio_data)
            sample_rate, audio_data = audio_file
            print(f"Получены аудиоданные: sample_rate={sample_rate}, форма={audio_data.shape}")
            
            # Сохраняем во временный файл
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
                temp_filename = tmp_file.name
                sf.write(temp_filename, audio_data, sample_rate)
            # Транскрибируем
            result = whisper_model.transcribe(temp_filename, language="ru")
            
            # Удаляем временный файл
            try:
                os.unlink(temp_filename)
            except:
                pass
                
            return result["text"]
        else:
            return f"Неподдерживаемый формат аудио: {type(audio_file)}"
        
    except Exception as e:
        print(f"Ошибка в transcribe_audio: {str(e)}")
        import traceback
        traceback.print_exc()
        return f"Ошибка распознавания речи: {str(e)}"

# LLM part (WIP)

In [7]:
def chat_with_llm(message): #, history):
    # Ollama's local endpoint (No API key needed)
    url = "http://localhost:11434/api/chat"
    payload = {
        "model": "deepseek-v3.1:671b-cloud",
        "messages": [{"role": "user", "content": message}],
        "stream": False
    }
    response = requests.post(url, json=payload)
    return response.json()['message']['content']

#gr.ChatInterface(chat_with_llm).launch(share=True)

In [46]:
def cleanup():
    """Очистка временных файлов"""
    files = glob.glob("response_*.wav") + glob.glob("temp_audio_*.wav")
    for f in files:
        try:
            os.remove(f)
            print(f"Удален файл: {f}")
        except:
            pass
        
    files = glob.glob("reresponse_*.wav") + glob.glob("temp_audio_*.wav")
    for f in files:
        try:
            os.remove(f)
            print(f"Удален файл: {f}")
        except:
            pass

In [ ]:
def clean_filename(text):
    # Создаем безопасное имя файла из текста
    # Берем только первые 50 символов и удаляем недопустимые символы
    safe_text = re.sub(r'[^\w\s-]', '', text[:50])
    safe_text = re.sub(r'[-\s]+', '_', safe_text)
    return safe_text or "response"

def chat_with_llm_waudio(message, history):
    # Ollama's local endpoint (No API key needed)
    url = "http://localhost:11434/api/chat"
    payload = {
        "model": "deepseek-v3.1:671b-cloud",
        "messages": [{"role": "user", "content": message}],
        "stream": False
    }
    try:
        response = requests.post(url, json=payload)
        text_response = response.json()['message']['content']
        
        # Создаем безопасное имя файла
        safe_filename = clean_filename(text_response)
        # Добавляем хеш для уникальности
        filename = f"response_{hash(message)}.wav"
        
        #audio_data = generate_tts_audio(text_response, "ru", 'ru_m.flac')
        #sf.write(filename, audio_data[1], samplerate=audio_data[0])
        
        return text_response, filename
    except Exception as e:
        return f"Ошибика: {str(e)}", None

In [32]:
# def respond(message, chat_history):
#         if not message.strip():
#             return "", chat_history, None
        
#         try:
#             text_response, audio_file = chat_with_llm_waudio(message, chat_history)
#             chat_history.append((message, text_response))
#             return "", chat_history, audio_file
#         except Exception as e:
#             error_msg = f"Ошибка: {str(e)}"
#             chat_history.append((message, error_msg))
#             return "", chat_history, None

# # Простой интерфейс с двумя способами ввода
# with gr.Blocks(title="Голосовой чат") as demo:
#     gr.Markdown("# 🎤 Голосовой чат-бот")
    
#     chatbot = gr.Chatbot(label="История чата", height=400)
    
#     with gr.Row():
#         with gr.Column(scale=2):
#             msg = gr.Textbox(
#                 label="Текстовый ввод",
#                 placeholder="Напишите сообщение...",
#                 lines=2
#             )
#         with gr.Column(scale=1):
#             audio_input = gr.Audio(
#                 label="Голосовой ввод",
#                 type="numpy",
#                 sources=["microphone"]
#             )
    
#     with gr.Row():
#         send_text = gr.Button("📝 Отправить текст", variant="primary")
#         send_voice = gr.Button("🎤 Отправить голос", variant="secondary")
#         clear = gr.Button("🗑️ Очистить")
    
#     audio_output = gr.Audio(label="Голосовой ответ", type="filepath")
    
#     def process_text(message, history):
#         if not message:
#             return "", history, None
#         response, audio = chat_with_llm_waudio(message, history)
#         history.append((f"✍️ {message}", response))
#         return "", history, audio
    
#     def process_voice(audio, history):
#         if not audio:
#             return None, history, None, ""
#         text = transcribe_audio(audio)
#         if text.startswith("Ошибка"):
#             history.append(("🎤 [голосовое сообщение]", f"❌ {text}"))
#             return None, history, None, ""
#         response, response_audio = chat_with_llm_waudio(text, history)
#         history.append((f"🎤 {text}", response))
#         return None, history, response_audio, ""
    
#     send_text.click(process_text, [msg, chatbot], [msg, chatbot, audio_output])
#     send_voice.click(process_voice, [audio_input, chatbot], [audio_input, chatbot, audio_output, msg])
#     clear.click(lambda: (None, None, None), None, [chatbot, audio_output, msg])

# demo.launch(share=True)

In [15]:
# gr.close_all()
# cleanup()

# Gradio UI

In [43]:
# Attempt to load the model at startup.
try:
    get_or_load_model()
except Exception as e:
    print(f"CRITICAL: Failed to load model on startup. Application may not function. Error: {e}")

In [ ]:
# def respond(message, chat_history):
#         if not message.strip():
#             return "", chat_history, None
        
#         try:
#             text_response, audio_file = chat_with_llm_waudio(message, chat_history)
#             chat_history.append((message, text_response))
#             return "", chat_history, audio_file
#         except Exception as e:
#             error_msg = f"Ошибка: {str(e)}"
#             chat_history.append((message, error_msg))
#             return "", chat_history, None
        


# Простой интерфейс с двумя способами ввода
with gr.Blocks(title="Голосовой чат") as demo:
    # СОХРАНЯЕМ СОСТОЯНИЕ для regenerate
    last_user_message = gr.State("")
    last_bot_response = gr.State("")
    last_audio_file = gr.State(None)
    
    gr.Markdown("# 🎤 Голосовой чат-бот")
    
    chatbot = gr.Chatbot(label="История чата", height=400)
    
    with gr.Row():
        with gr.Column(scale=2):
            msg = gr.Textbox(
                label="Текстовый ввод",
                placeholder="Напишите сообщение...",
                lines=2
            )           
            
                
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                label="Голосовой ввод",
                #type="numpy",
                sources=["upload", "microphone"],
                type="filepath",
            )
    
    with gr.Row():
        send_text = gr.Button("📝 Отправить текст", variant="primary")
        send_voice = gr.Button("🎤 Отправить голос", variant="secondary")
        clear = gr.Button("🗑️ Очистить")
        # Кнопка для применения настроек к последнему ответу
        regenerate_btn = gr.Button("🔄 Применить настройки к последнему ответу", variant="secondary")
    
    audio_output = gr.Audio(label="Голосовой ответ", type="filepath")
    
    with gr.Column(scale=1):
            # Панель настроек TTS
            gr.Markdown("### 🎛️ Настройки голоса")
            
            cfg_weight = gr.Slider(
                minimum=0.5,
                maximum=3.0,
                value=1.0,
                step=0.1,
                label="CFG Scale",
                info="Влияние классификатора (1.0 = нейтрально, выше = выразительнее)"
            )
            
            exaggeration = gr.Slider(
                minimum=0.5,
                maximum=2.5,
                value=1.0,
                step=0.1,
                label="Exaggeration",
                info="Усиление эмоций/артикуляции (1.0 = нормально)"
            )
            
            with gr.Accordion("More options", open=False):
                seed_num = gr.Number(value=0, label="Random seed (0 for random)")
                temp = gr.Slider(0.05, 5, step=.05, label="Temperature", value=.8)
                
    #cfg_weight.change([cfg_weight])
    #exaggeration.change([exaggeration])
       
    def process_text(message, history, exaggeration, cfg_weight):
        if not message:
            return "", history, None
        response, audio = chat_with_llm_waudio(message, history)
        
        audio_data = generate_tts_audio(response, "ru", 'ru_m.flac', exaggeration_input=exaggeration, cfgw_input=cfg_weight)
        sf.write(audio, audio_data[1], samplerate=audio_data[0])
        
        history.append((f"✍️ {message}", response))
        return "", history, audio, message, response, audio
    
    def process_voice(audio, history, exaggeration, cfg_weight):
        if not audio:
            return None, history, None, "", "", None
        text = transcribe_audio(audio)
        if text.startswith("Ошибка"):
            history.append(("🎤 [голосовое сообщение]", f"❌ {text}"))
            return None, history, None, "", "", None
        response, response_audio = chat_with_llm_waudio(text, history)
        audio_data = generate_tts_audio(response, "ru", 'ru_m.flac', exaggeration_input=exaggeration, cfgw_input=cfg_weight)
        sf.write(response_audio, audio_data[1], samplerate=audio_data[0])
        history.append((f"🎤 {text}", response))
        return None, history, response_audio, text, response, response_audio
    
    def regenerate(last_msg, last_resp, last_audio, history, cfg, exag):
        """Регенерация последнего ответа с новыми настройками"""
        if not last_msg or not last_resp:
            return history, None, "Нет сообщения для регенерации", last_msg, last_resp, None
        
        print(f"Регенерация ответа на: '{last_msg}' с cfg={cfg}, exag={exag}")
        
        # Опционально: можно снова запросить LLM или использовать сохраненный текст
        # Здесь используем сохраненный текст ответа для экономии времени
        response = last_resp  # или можно сделать новый запрос к LLM
        new_filename = f"reresponse_{hash(response)}.wav"
        audio = generate_tts_audio(response, "ru", 'ru_m.flac', exaggeration_input=exaggeration, cfgw_input=cfg_weight)
        sf.write(new_filename, audio[1], samplerate=audio[0])
        return history, new_filename, last_msg, response, new_filename
    
    def clear_all():
        """Очистка всего"""
        cleanup()
        return None, None, None, "", "", None
    
    send_text.click(process_text, [msg, chatbot, exaggeration, cfg_weight], [msg, chatbot, audio_output, last_user_message, last_bot_response, last_audio_file])
    send_voice.click(process_voice, [audio_input, chatbot, exaggeration, cfg_weight], [audio_input, chatbot, audio_output, last_user_message, last_bot_response, last_audio_file])
    #clear.click(lambda: (None, None, None), None, [chatbot, audio_output, msg])
    clear.click(
        clear_all,
        None,
        [chatbot, audio_output, audio_input, last_user_message, last_bot_response, last_audio_file]
    )
    # Кнопка регенерации - использует сохраненные состояния
    regenerate_btn.click(
        regenerate,
        inputs=[last_user_message, last_bot_response, last_audio_file, chatbot, cfg_weight, exaggeration],
        outputs=[chatbot, audio_output, last_user_message, last_bot_response, last_audio_file]
    )

demo.launch()

C:\Users\salni\AppData\Local\Temp\ipykernel_24604\4224011248.py:25: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="История чата", height=400)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Generating audio for text: 'Привет! 😊 Рад тебя видеть! 

Как у тебя дела? Чем ...'
Using audio prompt: ru_m.flac


C:\Users\salni\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\transformers\generation\configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_attentions` is. When `return_dict_in_generate` is not `True`, `output_attentions` is ignored.
  warnings.warn(
Sampling:  13%|█▎        | 133/1000 [00:05<00:34, 25.30it/s]



Audio generation complete.
Generating audio for text: 'Привет! 😊 Рад тебя видеть! 

Как у тебя дела? Чем ...'
Using audio prompt: ru_m.flac


...

C:\Users\salni\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\transformers\generation\configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_attentions` is. When `return_dict_in_generate` is not `True`, `output_attentions` is ignored.
  warnings.warn(
Sampling:  47%|████▋     | 470/1000 [00:20<00:23, 22.43it/s]



Audio generation complete.
Generating audio for text: 'Понимаю, что вы написали на русском, и я вижу в ва...'
Using audio prompt: ru_m.flac


Sampling:  38%|███▊      | 385/1000 [00:17<00:27, 22.62it/s]


Audio generation complete.
Удален файл: response_-5450859552519019307.wav
Удален файл: response_-5764939027147066022.wav


c:\Users\salni\CHATTER\chatterVenv\Lib\site-packages\gradio\blocks.py:1974: UserWarning: A function (regenerate) returned too many output values (needed: 5, returned: 6). Ignoring extra values.
    Output components:
        [chatbot, audio, state, state, state]
    Output values returned:
        [[], None, "Нет сообщения для регенерации", "", "", None]
  warnings.warn(


In [47]:
gr.close_all()
cleanup()

Удален файл: reresponse_6251450813270529851.wav
